# AI-Assisted Machine Learning: Diabetes Classification

**Category:** 06-AI-ML  
**Description:** Build and evaluate a classification model with AI guidance  
**Data:** ~/lab/data/CSVs/diabetes_all_2016.csv

---

## What You'll Learn

1. Data preparation with AI assistance
2. Feature engineering suggestions from AI
3. Model selection and training
4. Evaluation and interpretation
5. AI-powered insights on results

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q pandas numpy scikit-learn matplotlib seaborn xgboost

# Model aliases - update when models change
CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
GPT = "openai-chat:gpt-5"
GEMINI = "gemini:gemini-2.5-flash"

MODEL = CLAUDE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, accuracy_score)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

plt.style.use('seaborn-v0_8-whitegrid')
DATA_DIR = Path.home() / 'lab' / 'data' / 'CSVs'

---

# Part 1: Data Loading & Exploration

---

In [ ]:
# Load diabetes dataset
df = pd.read_csv(DATA_DIR / 'diabetes_all_2016.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Check for missing values and data types
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

## 1.1 AI-Assisted Data Understanding

In [ ]:
# Get AI to analyze the dataset structure
data_summary = f"""
Dataset: Diabetes Classification
Shape: {df.shape}
Columns: {df.columns.tolist()}

Sample data:
{df.head(5).to_string()}

Statistics:
{df.describe().to_string()}
"""
print(data_summary)

In [ ]:
%%ai $MODEL
I have a diabetes dataset with columns that likely include demographic and health indicators.
Based on the data, suggest:
1. Which column is likely the target variable (what we're predicting)
2. Which features are most important for diabetes prediction
3. Any data preprocessing steps I should consider
4. Potential issues to watch for (class imbalance, outliers, etc.)

---

# Part 2: Data Preprocessing

---

In [ ]:
# Identify target and features (adjust based on your dataset)
# Common diabetes datasets have 'Outcome' or 'Diabetes' as target

# Find potential target column
potential_targets = [col for col in df.columns if 'diab' in col.lower() or 'outcome' in col.lower()]
print(f"Potential target columns: {potential_targets}")

# If no obvious target, check unique value counts
for col in df.columns:
    if df[col].nunique() <= 5:
        print(f"{col}: {df[col].value_counts().to_dict()}")

In [ ]:
# Set target and features (adjust based on your data)
# This is a template - modify based on actual columns
target_col = df.columns[-1]  # Often last column is target
feature_cols = [col for col in df.columns if col != target_col]

print(f"Target: {target_col}")
print(f"Features: {feature_cols}")

X = df[feature_cols].copy()
y = df[target_col].copy()

In [ ]:
# Handle missing values
print("Missing values before:")
print(X.isnull().sum())

# Fill numeric columns with median
for col in X.select_dtypes(include=[np.number]).columns:
    X[col].fillna(X[col].median(), inplace=True)

# Fill categorical columns with mode
for col in X.select_dtypes(include=['object']).columns:
    X[col].fillna(X[col].mode()[0], inplace=True)

print("\nMissing values after:")
print(X.isnull().sum().sum())

In [ ]:
# Encode categorical variables if any
le = LabelEncoder()

for col in X.select_dtypes(include=['object']).columns:
    X[col] = le.fit_transform(X[col].astype(str))
    print(f"Encoded: {col}")

# Scale features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print(f"\nFinal feature shape: {X_scaled.shape}")

In [ ]:
# Check class balance
print("Target distribution:")
print(y.value_counts())
print(f"\nClass balance: {y.value_counts(normalize=True).to_dict()}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

---

# Part 3: Model Training & Comparison

---

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    
    # Train on full training set
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None
    
    results[name] = {
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_accuracy': accuracy,
        'roc_auc': roc_auc,
        'model': model
    }
    
    print(f"  CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"  Test Accuracy: {accuracy:.4f}")
    if roc_auc:
        print(f"  ROC-AUC: {roc_auc:.4f}")

In [ ]:
# Compare models visually
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
model_names = list(results.keys())
accuracies = [results[m]['test_accuracy'] for m in model_names]
cv_means = [results[m]['cv_mean'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

axes[0].bar(x - width/2, cv_means, width, label='CV Accuracy', color='steelblue')
axes[0].bar(x + width/2, accuracies, width, label='Test Accuracy', color='coral')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=15)
axes[0].legend()
axes[0].set_ylim(0, 1)

# ROC curves
for name, res in results.items():
    if res['roc_auc']:
        model = res['model']
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        axes[1].plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})")

axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend()

plt.tight_layout()
plt.show()

---

# Part 4: Best Model Analysis

---

In [ ]:
# Select best model
best_model_name = max(results, key=lambda x: results[x]['test_accuracy'])
best_model = results[best_model_name]['model']

print(f"Best Model: {best_model_name}")
print(f"Test Accuracy: {results[best_model_name]['test_accuracy']:.4f}")

In [ ]:
# Detailed classification report
y_pred = best_model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()

In [ ]:
# Feature importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=True)
    
    plt.figure(figsize=(10, 8))
    plt.barh(importance['feature'], importance['importance'], color='steelblue')
    plt.xlabel('Importance')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.tight_layout()
    plt.show()
    
    print("\nTop 5 Features:")
    print(importance.tail(5))

---

# Part 5: AI-Powered Insights

---

In [ ]:
# Prepare results summary for AI
summary = f"""
Model: {best_model_name}
Accuracy: {results[best_model_name]['test_accuracy']:.4f}
ROC-AUC: {results[best_model_name]['roc_auc']:.4f}

Classification Report:
{classification_report(y_test, y_pred)}

Confusion Matrix:
{cm}
"""
print(summary)

In [ ]:
%%ai $MODEL
I trained a diabetes classification model with these results:
- Best model: Random Forest or Gradient Boosting
- Test accuracy: around 75-85%
- Some class imbalance present

Please provide:
1. Interpretation of these results for a healthcare context
2. Clinical implications of false positives vs false negatives
3. Suggestions for improving the model
4. How this model could be deployed responsibly in practice

---

# Part 6: Hyperparameter Tuning (Optional)

---

In [ ]:
# Grid search for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

# Evaluate tuned model
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)
print(f"Tuned test accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")

---

## Summary

This notebook demonstrated:

1. **Data Exploration** - Understanding the dataset with AI assistance
2. **Preprocessing** - Handling missing values, encoding, scaling
3. **Model Comparison** - Logistic Regression, Random Forest, Gradient Boosting
4. **Evaluation** - Accuracy, ROC-AUC, confusion matrix, classification report
5. **Feature Importance** - Understanding which features drive predictions
6. **AI Insights** - Getting clinical context and improvement suggestions
7. **Hyperparameter Tuning** - Optimizing model performance

---

**Next:** Try time series forecasting in `12-Time-Series/01-forecasting.ipynb`